In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from umap import UMAP
import matplotlib.pyplot as plt
import pandas as pd
from scipy import linalg
import torch
from sklearn.manifold import TSNE
import umap.plot
from sklearn.preprocessing  import MinMaxScaler
from Bio import SeqIO

/doctorai/marinafr/progs/miniconda3/envs/airr_atlas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.
    Params:
    -- mu1:    The mean of the activations of preultimate layer of the
               CHEMNET (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2:    The mean of the activations of preultimate layer of the
               CHEMNET (like returned by the function 'get_predictions')
               for real samples.
    -- sigma1: The covariance matrix of the activations of preultimate layer
               of the CHEMNET (like returned by the function 'get_predictions')
               for generated samples.
    -- sigma2: The covariance matrix of the activations of preultimate layer
               of the CHEMNET (like returned by the function 'get_predictions')
               for real samples.
    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert (
        mu1.shape == mu2.shape
    ), "Training and test mean vectors have different lengths"
    assert (
        sigma1.shape == sigma2.shape
    ), "Training and test covariances have different dimensions"

    diff = mu1 - mu2

    # product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            #raise ValueError("Imaginary component {}".format(m))
            print("Imaginary component {}".format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * tr_covmean

In [3]:
def get_w2(act1, act2):

    """Calculate w2 between two sets

    Args:
        act1: First set
        act2: Second set

    Returns:
        float: The FCD score
    """

    mu1 = np.mean(act1, axis=0)
    sigma1 = np.cov(act1.T)

    mu2 = np.mean(act2, axis=0)
    sigma2 = np.cov(act2.T)

    fcd_score = calculate_frechet_distance(
        mu1=mu1, mu2=mu2, sigma1=sigma1, sigma2=sigma2
    )

    return fcd_score

In [14]:
model_name = 'antiberta2'

if model_name == 'antiberta2':
    short_model_name = 'ab2'
else: 
    short_model_name = model_name

data1 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()
data2 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()

print(len(data2))

800000


In [15]:
# Split the data into two halves
data1 = data1[:len(data1) // 2]
data2 = data2[:len(data2) // 2]

splits = [int(x) for x in [10**4, 3*10**4, 5*10**4, 10**5, 1.5*10**5, 2*10**5, 3*10**5, 4*10**5-100]]
w2_data = []

num_repetitions = 10

for repetition in range(num_repetitions):
    print(repetition+1)
    w2_repetition = []

    # Shuffle data2 to ensure randomization
    np.random.shuffle(data2)

    for batch_size in splits:
        print(batch_size)
        max_start = len(data2) - batch_size
        if max_start <= 0:
            print('break')
            break
        start = np.random.randint(0, max_start)
        batch = data2[start:start + batch_size]
        w2_repetition.append(get_w2(batch, data1))
        print(get_w2(batch, data1))

    w2_data.append(w2_repetition)

# Create a DataFrame for boxplot
df = pd.DataFrame(w2_data, columns=splits)

df.to_csv(f"baseline_1_patient_{short_model_name}.csv", index=False)

df

1
10000
1.1756779756121887
30000
0.516964897085245
50000
0.3722154971055147
100000
0.28350441121745007
150000
0.24973930886983453
200000
0.23761153865370943
300000
0.22459193317121162
399900
0.21314666603126398
2
10000
1.100439787496839
30000
0.5003850196723647
50000
0.3980723876119896
100000
0.28460993310193317
150000
0.2528644764023511
200000
0.23577107251981033
300000
0.22094087705846732
399900
0.2131389055202817
3
10000
1.149011697725598
30000
0.534190130018203
50000
0.36020653617293874
100000
0.28129423425201594
150000
0.24170159481622022
200000
0.2349434737931233
300000
0.218373038088032
399900
0.2130318526151882
4
10000
1.135438076638593
30000
0.5035352003398543
50000
0.36426928225182564
100000
0.2895574738237201
150000
0.24931910858674655
200000
0.23232918651109458
300000
0.21986105416254986
399900
0.21315680006597404
5
10000
1.121937080455325
30000
0.4970942424246232
50000
0.38014778628615886
100000
0.28074984878145415
150000
0.25416264287383683
200000
0.23980309929396526
3000

,10000,30000,50000,100000,150000,200000,300000,399900
0,1.175678,0.516965,0.372215,0.283504,0.249739,0.237612,0.224592,0.213147
1,1.100440,0.500385,0.398072,0.284610,0.252864,0.235771,0.220941,0.213139
2,1.149012,0.534190,0.360207,0.281294,0.241702,0.234943,0.218373,0.213032
3,1.135438,0.503535,0.364269,0.289557,0.249319,0.232329,0.219861,0.213157
4,1.121937,0.497094,0.380148,0.280750,0.254163,0.239803,0.225091,0.213100
5,1.148002,0.511794,0.369945,0.288047,0.260891,0.231760,0.225384,0.213067
6,1.118634,0.515481,0.380136,0.299621,0.253735,0.232154,0.218560,0.213188
7,1.084692,0.502604,0.376191,0.289100,0.238919,0.236020,0.218047,0.213044
8,1.084263,0.499454,0.372349,0.282561,0.255359,0.236980,0.220388,0.213066
9,1.153348,0.527879,0.355870,0.282829,0.256133,0.238117,0.220493,0.213171
